# Phase 1 — curation report

Reads `data/curated/report_dataset.jsonl` (every sample processed, accepted or not) and breaks down what got filtered out and why. This is the evidence-of-rigor artifact for the README: not just "I cleaned the data", but how much was discarded, for which reasons, across which languages, and in what prompt format.

Dependencies (`pandas`, `matplotlib`, `seaborn`) are in the `dev` dependency group in `pyproject.toml` — run `uv sync` rather than `pip install`ing them inline, so the notebook's environment stays in sync with the rest of the project.

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.data_curation.curation_config import REPORT_PATH

sys.path.insert(0, str(Path.cwd().parent))

metadata = []
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        metadata.append(sample["metadata"])

df = pd.DataFrame(metadata)
total = len(df)
accepted = (df["status"] == "accepted").sum()
rejected = total - accepted

print(f"Samples analyzed: {total}")
print(f"Accepted: {accepted} ({accepted / total:.1%})")
print(f"Rejected: {rejected} ({rejected / total:.1%})")

## Accepted vs. rejected

In [ ]:
plt.pie(
    [accepted, rejected],
    labels=["Accepted", "Rejected"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#F44336"],
)
plt.title("Accepted vs. rejected")
plt.show()

## Rejection reasons

Current status categories: `rejected_code_smell` (internal copy-paste), `rejected_syntax`, `rejected_lint`, `rejected_complexity`, `rejected_duplicate` (cross-sample near-duplicate). Grouping by these, not by the raw `error` text — error messages are mostly unique strings (different line numbers, different duplicate ids) and won't aggregate into anything readable.

In [ ]:
df_rejected = df[df["status"] != "accepted"]
order = df_rejected["status"].value_counts().index
sns.countplot(data=df_rejected, x="status", order=order)
plt.title("Rejection reasons")
plt.xticks(rotation=20)
plt.show()

## Acceptance rate by language

Worth checking on its own: languages without a lint signal yet (java, c_sharp, javascript, typescript — see README) are only gated on syntax + complexity, so their accept rate isn't directly comparable to python/cpp's. A language with a surprisingly low accept rate here is worth a manual look before assuming the gates are working the same way everywhere.

In [ ]:
lang_accept = df.groupby("language")["status"].apply(lambda s: (s == "accepted").mean())
lang_accept.sort_values(ascending=False).plot(kind="bar", color="#4CAF50")
plt.ylabel("Acceptance rate")
plt.title("Acceptance rate by language")
plt.xticks(rotation=20)
plt.show()

## Prompt format mix (accepted samples only)

`edit` (in-context: existing code + a short instruction) vs. `new_file` (write-from-spec, no context — closer to how HumanEval/MBPP prompt a model, see README). Worth checking this isn't overwhelmingly one or the other before moving to Phase 2.

In [ ]:
df_accepted = df[df["status"] == "accepted"]
df_accepted["prompt_type"].value_counts().plot(kind="bar", color="#2196F3")
plt.title("Prompt format among accepted samples")
plt.xticks(rotation=0)
plt.show()

## Example errors per rejection reason

A few representative examples per category, rather than a `value_counts()` on raw error text (which fragments into near-unique buckets).

In [ ]:
for status, group in df_rejected.groupby("status"):
    print(f"\n{status} ({len(group)} samples) — example errors:")
    sample_n = min(3, len(group))
    for err in group["error"].dropna().sample(sample_n, random_state=0):
        print(f"  - {err}")